## Indikatoren für XAU berechnen und exportieren

In [15]:
import pandas as pd
import os
from ta.momentum import RSIIndicator
from ta.trend import MACD, SMAIndicator

# 📥 1. Daten laden
df = pd.read_csv("../dataset/output/XAUUSD_1d_analysis_clean.csv", parse_dates=["date_xau"])
df = df.sort_values("date_xau")

# 🧹 2. Relevante Spalten in float konvertieren
for col in ["close_xau", "open_xau", "high_xau", "low_xau", "volume_xau"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Entferne Zeilen mit fehlendem Schlusskurs
df = df.dropna(subset=["close_xau"]).copy()

# 📊 3. Technische Indikatoren berechnen
# RSI
rsi = RSIIndicator(close=df["close_xau"], window=14)
df["RSI"] = rsi.rsi()

# MACD
macd = MACD(close=df["close_xau"], window_slow=26, window_fast=12, window_sign=9)
df["MACD"] = macd.macd()

# MA20
sma = SMAIndicator(close=df["close_xau"], window=20)
df["MA_20"] = sma.sma_indicator()

# Trendlinie (z. B. identisch mit MA_20)
df["Trendline"] = df["MA_20"]

# 🧾 4. Finale Spaltenauswahl und Umbenennung
df_out = df[[
    "date_xau", "open_xau", "high_xau", "low_xau", "close_xau",
    "RSI", "MACD", "MA_20", "Trendline", "volume_xau"
]]
df_out.columns = [
    "timestamp", "open", "high", "low", "close",
    "RSI", "MACD", "MA_20", "Trendline", "volume"
]

# 🔧 5. Runden und fehlende Werte als leere Felder
df_out = df_out.round(5).fillna("")

# ❗ 6. Falls Datei schon existiert → löschen
output_path = "../dataset/output/XAUUSD_technical_1d_clean.csv"
if os.path.exists(output_path):
    os.remove(output_path)

# 📤 7. Speichern
df_out.to_csv(output_path, index=False, sep=",")

## Indikatoren für BTC berechnen und exportieren

In [17]:
import pandas as pd
import os
from ta.momentum import RSIIndicator
from ta.trend import MACD, SMAIndicator

# 📥 1. Daten laden
df = pd.read_csv("../dataset/output/BTCUSD_1d_analysis_clean.csv", parse_dates=["date_btc"])
df = df.sort_values("date_btc")

# 🧹 2. Relevante Spalten in float konvertieren
for col in ["close_btc", "open_btc", "high_btc", "low_btc", "volume_usd_btc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Entferne Zeilen mit fehlendem Schlusskurs
df = df.dropna(subset=["close_btc"]).copy()

# 📊 3. Technische Indikatoren berechnen
# RSI
rsi = RSIIndicator(close=df["close_btc"], window=14)
df["RSI"] = rsi.rsi()

# MACD
macd = MACD(close=df["close_btc"], window_slow=26, window_fast=12, window_sign=9)
df["MACD"] = macd.macd()

# MA20
sma = SMAIndicator(close=df["close_btc"], window=20)
df["MA_20"] = sma.sma_indicator()

# Trendlinie (z. B. identisch mit MA_20)
df["Trendline"] = df["MA_20"]

# 🧾 4. Finale Spaltenauswahl und Umbenennung
df_out = df[[
    "date_btc", "open_btc", "high_btc", "low_btc", "close_btc",
    "RSI", "MACD", "MA_20", "Trendline", "volume_usd_btc"
]]
df_out.columns = [
    "timestamp", "open", "high", "low", "close",
    "RSI", "MACD", "MA_20", "Trendline", "volume"
]

# 🔧 5. Runden und fehlende Werte als leere Felder
df_out = df_out.round(5).fillna("")

# ❗ 6. Falls Datei schon existiert → löschen
output_path = "../dataset/output/BTCUSD_technical_1d_clean.csv"
if os.path.exists(output_path):
    os.remove(output_path)

# 📤 7. Speichern
df_out.to_csv(output_path, index=False, sep=",")


## Merge technische Indikatoren für BTC und XAU

In [19]:
import pandas as pd
import os

# 📥 1. Dateien laden
btc_path = "../dataset/output/BTCUSD_technical_1d_clean.csv"
xau_path = "../dataset/output/XAUUSD_technical_1d_clean.csv"

btc = pd.read_csv(btc_path, parse_dates=["timestamp"])
xau = pd.read_csv(xau_path, parse_dates=["timestamp"])

# 📆 2. Merge auf Basis des Zeitstempels
merged = pd.merge(btc, xau, on="timestamp", how="inner", suffixes=("_btc", "_xau"))

# 🧽 3. Sortierung und Bereinigung
merged = merged.sort_values("timestamp").reset_index(drop=True)

# 🔍 3.1 Entferne Zeilen mit fehlenden technischen Indikatoren (z. B. am Anfang)
merged = merged.dropna(subset=["RSI_btc", "MACD_btc", "RSI_xau", "MACD_xau"]).copy()

# ❗ 4. Falls Datei schon existiert → löschen
output_path = "../dataset/output/final_merged_technical_dataset.csv"
if os.path.exists(output_path):
    os.remove(output_path)

# 💾 5. Speichern
merged.to_csv(output_path, index=False, sep=",")

print("✅ Merge abgeschlossen und gespeichert unter:", output_path)


✅ Merge abgeschlossen und gespeichert unter: ../dataset/output/final_merged_technical_dataset.csv
